# 02 — Analisis Exploratorio de Datos (EDA)

**Objetivo:** Explorar el dataset de nacimientos limpio para comprender la distribucion de las variables objetivo y generar hipotesis para el modelado.

Ejecute primero `01_limpieza_datos.ipynb`.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'nac2018_cleaned.csv'
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

In [ ]:
df = pd.read_csv(PROCESSED_PATH)
print(f'Dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas')
df.head(3)

## 1. Distribucion de variables objetivo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, target, title, labels in zip(
    axes,
    ['CESAREA', 'BAJO_PESO'],
    ['Tipo de Parto', 'Peso al Nacer'],
    [['Espontaneo', 'Cesarea'], ['Normal (>=2500g)', 'Bajo peso (<2500g)']],
):
    counts = df[target].value_counts().sort_index()
    bars = ax.bar(labels, counts.values, color=['#4CAF50', '#F44336'],
                  alpha=0.85, edgecolor='white')
    ax.set_title(title, fontweight='bold', pad=12)
    ax.set_ylabel('Numero de registros')
    for bar, count in zip(bars, counts.values):
        pct = count / len(df) * 100
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 500,
                f'{count:,}\n({pct:.1f}%)',
                ha='center', va='bottom', fontsize=10)

plt.suptitle('Distribucion de Variables Objetivo',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 2. Edad materna y resultados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, target, title in zip(
    axes,
    ['CESAREA', 'BAJO_PESO'],
    ['Edad materna vs Cesarea', 'Edad materna vs Bajo Peso'],
):
    for val, color, label in [(0, '#4CAF50', 'No'), (1, '#F44336', 'Si')]:
        df[df[target] == val]['EDAD_MADRE'].dropna().plot.kde(
            ax=ax, color=color, label=label, alpha=0.7, linewidth=2)
    ax.axvline(18, color='orange', linestyle='--', alpha=0.7, label='18 anos')
    ax.axvline(35, color='purple', linestyle='--', alpha=0.7, label='35 anos')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Edad de la madre (anos)')
    ax.legend(title='Resultado')

plt.suptitle('Distribucion de Edad Materna por Resultado',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
df['GRUPO_EDAD'] = pd.cut(
    df['EDAD_MADRE'],
    bins=[0, 17, 24, 34, 45, 100],
    labels=['<18', '18-24', '25-34', '35-45', '>45'],
)
edad_stats = (df.groupby('GRUPO_EDAD', observed=True)
               [['CESAREA', 'BAJO_PESO']].mean() * 100)
print('Tasa (%) por grupo de edad materna:')
edad_stats.round(1)

## 3. Semanas de gestacion (T_GES)

In [ ]:
print('Distribucion de T_GES:')
print(df['T_GES'].describe())
print('\nValue counts ordenados:')
print(df['T_GES'].value_counts().sort_index())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
tasa_ces = df.groupby('T_GES')['CESAREA'].mean() * 100
tasa_bp = df.groupby('T_GES')['BAJO_PESO'].mean() * 100
ax.plot(tasa_ces.index, tasa_ces.values, 'o-', color='#F44336', label='Cesarea %')
ax.plot(tasa_bp.index, tasa_bp.values, 's-', color='#FF9800', label='Bajo Peso %')
ax.set_xlabel('T_GES')
ax.set_ylabel('Tasa (%)')
ax.set_title('Tasas por Semanas de Gestacion', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Acceso a salud y area de residencia

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

area_map = {1: 'Urbana', 2: 'Centro Poblado', 3: 'Rural', -1: 'Desconocido'}
regimen_map = {1: 'Contributivo', 2: 'Subsidiado', -1: 'No asegurado'}

for ax, col, mapping, title in zip(
    axes,
    ['AREA_RES', 'IDCLASADMI'],
    [area_map, regimen_map],
    ['Area de Residencia', 'Regimen de Salud'],
):
    stats = df.groupby(col)[['CESAREA', 'BAJO_PESO']].mean() * 100
    stats.index = [mapping.get(i, str(i)) for i in stats.index]
    stats.plot(kind='bar', ax=ax,
               color=['#F44336', '#FF9800'], alpha=0.85, edgecolor='white')
    ax.set_title(f'Tasas por {title}', fontweight='bold')
    ax.set_ylabel('Tasa (%)')
    ax.legend(['Cesarea', 'Bajo Peso'])
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

plt.suptitle('Tasas segun Acceso a Salud', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Control prenatal y paridad

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

cons_valid = df[df['NUMCONSUL'].between(0, 20)]
tasa_ces = cons_valid.groupby('NUMCONSUL')['CESAREA'].mean() * 100
tasa_bp = cons_valid.groupby('NUMCONSUL')['BAJO_PESO'].mean() * 100
axes[0].plot(tasa_ces.index, tasa_ces.values, 'o-', color='#F44336', label='Cesarea %')
axes[0].plot(tasa_bp.index, tasa_bp.values, 's-', color='#FF9800', label='Bajo Peso %')
axes[0].axvline(4, color='gray', linestyle='--', alpha=0.6, label='Min recomendado')
axes[0].set_xlabel('Numero de consultas prenatales')
axes[0].set_ylabel('Tasa (%)')
axes[0].set_title('Control Prenatal vs Resultado', fontweight='bold')
axes[0].legend()

mul_map = {1: 'Unico', 2: 'Gemelar', 3: 'Triple+'}
mul_df = df[df['MUL_PARTO'].isin([1, 2, 3])]
mul_stats = mul_df.groupby('MUL_PARTO')[['CESAREA', 'BAJO_PESO']].mean() * 100
mul_stats.index = [mul_map[i] for i in mul_stats.index]
mul_stats.plot(kind='bar', ax=axes[1],
               color=['#F44336', '#FF9800'], alpha=0.85, edgecolor='white')
axes[1].set_title('Tipo de Gestacion vs Resultado', fontweight='bold')
axes[1].set_ylabel('Tasa (%)')
axes[1].legend(['Cesarea', 'Bajo Peso'])
plt.setp(axes[1].get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## 6. Top departamentos

In [ ]:
dept_names = {
    11: 'Bogota', 5: 'Antioquia', 76: 'Valle', 8: 'Atlantico',
    13: 'Bolivar', 25: 'Cundinamarca', 68: 'Santander', 52: 'Narino',
    23: 'Cordoba', 19: 'Cauca', 15: 'Boyaca', 73: 'Tolima',
    47: 'Magdalena', 41: 'Huila', 20: 'Cesar'
}
top_depts = df['CODPTORE'].value_counts().nlargest(15).index.tolist()
dept_df = (
    df[df['CODPTORE'].isin(top_depts)]
    .groupby('CODPTORE')[['CESAREA', 'BAJO_PESO']]
    .mean() * 100
)
dept_df.index = [dept_names.get(i, str(i)) for i in dept_df.index]
dept_df = dept_df.sort_values('CESAREA', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
dept_df.plot(kind='barh', ax=ax,
             color=['#F44336', '#FF9800'], alpha=0.85, edgecolor='white')
ax.set_title('Tasas por Departamento de Residencia (Top 15)', fontweight='bold')
ax.set_xlabel('Tasa (%)')
ax.legend(['Cesarea', 'Bajo Peso'])
plt.tight_layout()
plt.show()

## 7. Matriz de correlacion

In [ ]:
from src.preprocessing import NUMERIC_FEATURES

corr_cols = NUMERIC_FEATURES + ['CESAREA', 'BAJO_PESO']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, mask=mask, ax=ax,
    linewidths=0.5, square=True, cbar_kws={'shrink': 0.8},
)
ax.set_title('Matriz de Correlacion', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 8. Resumen

In [ ]:
from src.preprocessing import FEATURES, NUMERIC_FEATURES

print('=' * 60)
print('RESUMEN EDA')
print('=' * 60)
print(f'Total registros validos: {len(df):,}')
print(f"Tasa de cesarea:        {df['CESAREA'].mean():.1%}")
print(f"Tasa de bajo peso:      {df['BAJO_PESO'].mean():.1%}")
print(f"Edad materna media:     {df['EDAD_MADRE'].mean():.1f} anos")
print(f'Nulos en features:      {df[FEATURES].isnull().sum().sum()}')
print('\nDescriptivas numericas:')
df[NUMERIC_FEATURES].describe().round(2)